# AIC2026 Trial P1 — live validation

Default mode is `B0_SAFE`: official 24-query package → deterministic compiler → current B0 → fail-closed CSV/ZIP validator. No GT is mounted or opened.

In [ ]:
import json, os, re, shutil, subprocess, sys
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
REPO_URL=os.environ.get('AIC_REPO_URL','https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF=os.environ.get('AIC_REPO_REF','TRIAGEEG')
REPO_DIR=Path(os.environ.get('AIC_REPO_DIR','/kaggle/working/AIC2026_TeamPTK_SGU'))
REFRESH_REPO=os.environ.get('AIC_REFRESH_REPO','0')=='1'
TRIAL_MODE=os.environ.get('AIC_TRIAL_MODE','B0_SAFE').upper()
TRIAL_INPUT=Path(os.environ.get('AIC_TRIAL_P1_ROOT','/kaggle/input/datasets/irthn1311/thunghiem-bo-de-thi'))
DATA_INPUT=Path(os.environ.get('AIC_DATA_ROOT','/kaggle/input/datasets/nadkli/dataset-aic'))
STAGE1_INPUT=Path(os.environ.get('AIC_STAGE1_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle'))
STAGE1B_INPUT=Path(os.environ.get('AIC_STAGE1B_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports'))
STAGE1E_INPUT=Path(os.environ.get('AIC_STAGE1E_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze'))
CLIP_INPUT=Path(os.environ.get('AIC_CLIP_ROOT','/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32'))
OPUS_INPUT=Path(os.environ.get('AIC_OPUS_ROOT','/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en'))
OUTPUT_ROOT=Path('/kaggle/working/trial_p1_B0_SAFE_artifacts')
SUBMISSION_ZIP=Path('/kaggle/working/trial_p1_B0_SAFE_submission.zip')
BUNDLE_ZIP=Path('/kaggle/working/trial_p1_B0_SAFE_bundle.zip')
WORK_ROOT=Path('/kaggle/working/trial_p1_work')
if TRIAL_MODE!='B0_SAFE': raise RuntimeError('This immediate notebook is bounded to B0_SAFE; FULL/SAFE require finalized v1.1 evidence and must not silently fall back')
for target in (OUTPUT_ROOT,WORK_ROOT):
    if target.exists(): shutil.rmtree(target)
for target in (SUBMISSION_ZIP,BUNDLE_ZIP): target.unlink(missing_ok=True)
print({'mode':TRIAL_MODE,'required_inputs':{'official_trial_package':str(TRIAL_INPUT),'raw_dataset':str(DATA_INPUT),'stage1_exact_index':str(STAGE1_INPUT),'stage1b_verified_contract':str(STAGE1B_INPUT),'stage1e_language_contract':str(STAGE1E_INPUT),'openai_clip_offline_asset':str(CLIP_INPUT),'opus_mt_vi_en_offline_asset':str(OPUS_INPUT)},'internet_required':'ONLY_FOR_GIT_CLONE_OR_EXPLICIT_REFRESH','model_download_required':False,'submission_zip':str(SUBMISSION_ZIP),'bundle_zip':str(BUNDLE_ZIP)})


In [ ]:
def git_result(*args,cwd=None): return subprocess.run(['git',*args],cwd=cwd,capture_output=True,text=True,check=False)
def git(*args,cwd=None):
    result=git_result(*args,cwd=cwd)
    if result.returncode: raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()
if REPO_DIR.exists() and not (REPO_DIR/'.git').is_dir() and any(REPO_DIR.iterdir()): raise RuntimeError(f'{REPO_DIR} is not a Git checkout')
if not (REPO_DIR/'.git').is_dir():
    REPO_DIR.parent.mkdir(parents=True,exist_ok=True); git('clone','--filter=blob:none','--no-checkout',REPO_URL,str(REPO_DIR))
target=None
if not REFRESH_REPO:
    for candidate in (REPO_REF,f'origin/{REPO_REF}'):
        probe=git_result('rev-parse','--verify',f'{candidate}^{{commit}}',cwd=REPO_DIR)
        if probe.returncode==0: target=probe.stdout.strip(); break
if target is None: git('fetch','--no-tags','origin',REPO_REF,cwd=REPO_DIR); target='FETCH_HEAD'
git('checkout','--detach',target,cwd=REPO_DIR)
HEAD=git('rev-parse','HEAD',cwd=REPO_DIR)
if not (REPO_DIR/'src/triage_eg/trial_p1/runner.py').is_file(): raise RuntimeError('Resolved ref lacks Trial P1 implementation')
sys.path.insert(0,str(REPO_DIR/'src'))
print({'source_ref':REPO_REF,'HEAD':HEAD,'git_status':git('status','--short',cwd=REPO_DIR) or 'CLEAN'})


In [ ]:
def mount_candidates(hint):
    names=(hint.name,hint.name.replace('_','-'),hint.name.replace('-','_'))
    roots=[hint,*[Path('/kaggle/input')/name for name in names],*[Path('/kaggle/input/datasets/irthn1311')/name for name in names],*[Path('/kaggle/input/datasets/nadkli')/name for name in names]]
    return sorted({path.resolve() for path in roots if path.exists()})
def one_mount(hint):
    found=mount_candidates(hint)
    if len(found)!=1: raise RuntimeError(f'Expected one mount for {hint}; found {found}')
    return found[0]
def find_unique(root,name):
    found=sorted(root.rglob(name))
    if len(found)!=1: raise RuntimeError(f'Expected exactly one {name} below {root}; found {found}')
    return found[0]
TRIAL_MOUNT=one_mount(TRIAL_INPUT)
trial_zips=sorted(TRIAL_MOUNT.rglob('THUNGHIEM-bo-de-thi.zip'))
if len(trial_zips)==1: TRIAL_ZIP=trial_zips[0]; TRIAL_SOURCE_MODE='ORIGINAL_ZIP'
elif not trial_zips:
    txts=sorted(TRIAL_MOUNT.rglob('query-p1-*-*.txt'))
    parents={path.parent.resolve() for path in txts}
    if len(txts)!=24 or len(parents)!=1: raise RuntimeError(f'Official Trial P1 ZIP absent and expanded package is not one exact 24-file root: {len(txts)}, {parents}')
    TRIAL_ZIP=WORK_ROOT/'THUNGHIEM-bo-de-thi.zip'; TRIAL_ZIP.parent.mkdir(parents=True,exist_ok=True)
    with ZipFile(TRIAL_ZIP,'w',ZIP_DEFLATED) as archive:
        for path in txts: archive.write(path,path.name)
    TRIAL_SOURCE_MODE='KAGGLE_EXPANDED_PACKAGE_REPACKED_BYTE_EXACT'
else: raise RuntimeError(f'Ambiguous Trial P1 ZIPs: {trial_zips}')
DATASET_ROOT=one_mount(DATA_INPUT); STAGE1_MOUNT=one_mount(STAGE1_INPUT); STAGE1B_MOUNT=one_mount(STAGE1B_INPUT); STAGE1E_MOUNT=one_mount(STAGE1E_INPUT); CLIP_MOUNT=one_mount(CLIP_INPUT); OPUS_MOUNT=one_mount(OPUS_INPUT)
print({'trial_zip':str(TRIAL_ZIP),'trial_source_mode':TRIAL_SOURCE_MODE,'raw_mount':str(DATASET_ROOT),'stage1_mount':str(STAGE1_MOUNT),'stage1b_mount':str(STAGE1B_MOUNT),'stage1e_mount':str(STAGE1E_MOUNT),'clip_mount':str(CLIP_MOUNT),'opus_mount':str(OPUS_MOUNT)})


In [ ]:
from aic2026_eval.io import write_json, write_jsonl
from triage_eg.trial_p1 import compile_queries, parse_trial_zip
MANIFEST=parse_trial_zip(TRIAL_ZIP); COMPILED=compile_queries(MANIFEST)
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
write_json(OUTPUT_ROOT/'trial_p1_query_manifest.json',MANIFEST)
write_jsonl(OUTPUT_ROOT/'trial_p1_query_plans.jsonl',COMPILED)
TRAKE={row['query_id']:{'event_count':row['event_count'],'raw_event_labels':row['raw_event_labels'],'internal_event_ids':[event['event_id'] for event in row['events']]} for row in MANIFEST['queries'] if row['task']=='TRAKE'}
ROUTING=[{'query_id':plan['query_id'],'task':plan['task'],'answer_type':plan['answer_type'],'routing':plan['routing'],'retrieval_variants':plan['retrieval_variants']} for plan in COMPILED]
write_json(OUTPUT_ROOT/'trial_p1_routing_plan.json',ROUTING)
assert TRAKE['query-p1-18-trake']['raw_event_labels']==['E1','E2','E2','E4'] and TRAKE['query-p1-18-trake']['event_count']==4
test_env=dict(os.environ); test_env['PYTHONPATH']=os.pathsep.join([str(REPO_DIR/'src'),test_env.get('PYTHONPATH','')]); test_env['AIC_TRIAL_P1_ZIP']=str(TRIAL_ZIP)
test=subprocess.run([sys.executable,'-m','pytest','tests/unit/trial_p1','-q'],cwd=REPO_DIR,env=test_env,capture_output=True,text=True)
if test.returncode: raise RuntimeError(test.stdout+'\n'+test.stderr)
print({'manifest':{'total':MANIFEST['query_count'],**MANIFEST['task_counts']},'trake':TRAKE,'warning':'query-p1-18-trake repeats raw E2; four ordinal events preserved','routing_plan_count':len(ROUTING),'tests':test.stdout.splitlines()[-1],'GT_OPENED':False})


In [ ]:
from triage_eg.e2e1.pipeline import CanonicalTriagePipeline
from triage_eg.retrieval.stage1b.adapters.openai_clip_official import materialize_kaggle_expanded_tokenizer, resolve_official_asset_paths
from triage_eg.retrieval.stage1b.inputs import resolve_stage1_root
from triage_eg.retrieval.stage1d.inputs import resolve_input_root
from triage_eg.retrieval.stage2 import OperationalRetrievalRuntime, config_from_yaml
from triage_eg.trial_p1 import run_b0_safe
STAGE1_ROOT=resolve_stage1_root(STAGE1_MOUNT,search_root=None,materialize_root=WORK_ROOT/'stage1')
STAGE1B_ROOT,_=resolve_input_root(STAGE1B_MOUNT,required=('stage1b_summary.json','encoder/selected_encoder_contract.json','encoder/runtime_adapter_manifest.json'),materialize_root=WORK_ROOT/'stage1b',search_root=None,archive_keyword='stage1b')
STAGE1E_ROOT,_=resolve_input_root(STAGE1E_MOUNT,required=('stage1e_summary.json','language_path_contract.json'),materialize_root=WORK_ROOT/'stage1e',search_root=None,archive_keyword='stage1e')
CLIP_ROOT,_=resolve_input_root(CLIP_MOUNT,required=('checkpoint/ViT-B-32.pt','manifests/asset_manifest.json'),materialize_root=WORK_ROOT/'clip',search_root=None,archive_keyword='clip')
OPUS_ROOT,_=resolve_input_root(OPUS_MOUNT,required=('model/config.json','manifests/asset_manifest.json'),materialize_root=WORK_ROOT/'opus',search_root=None,archive_keyword='opus')
paths=resolve_official_asset_paths(CLIP_ROOT); clip_source,_=materialize_kaggle_expanded_tokenizer(paths.source_root,WORK_ROOT/'shared_openai_clip_source'); os.environ['AIC_OPENAI_CLIP_SOURCE_ROOT']=str(clip_source)
config=config_from_yaml(REPO_DIR/'configs/retrieval/stage2_operational_runtime_gpu.yaml',stage1_root=STAGE1_ROOT,stage1b_root=STAGE1B_ROOT,stage1e_root=STAGE1E_ROOT,clip_asset_root=CLIP_ROOT,translator_asset_root=OPUS_ROOT,output_root=WORK_ROOT/'runtime_b0_safe',stage1d_config=REPO_DIR/'configs/retrieval/stage1d_translation_ablation.yaml',build_git_commit=HEAD)
runtime=OperationalRetrievalRuntime(config).load(); pipeline=CanonicalTriagePipeline(runtime,DATASET_ROOT)
RESULT=run_b0_safe(pipeline,COMPILED,OUTPUT_ROOT,SUBMISSION_ZIP)
print(RESULT)


In [ ]:
write_json(OUTPUT_ROOT/'run_summary.json',{'HEAD':HEAD,'mode':TRIAL_MODE,'trial_source_mode':TRIAL_SOURCE_MODE,'resolved_inputs':{'trial_zip':str(TRIAL_ZIP),'raw':str(DATASET_ROOT),'stage1':str(STAGE1_ROOT),'stage1b':str(STAGE1B_ROOT),'stage1e':str(STAGE1E_ROOT),'clip':str(CLIP_ROOT),'opus':str(OPUS_ROOT)},'result':RESULT,'GT_OPENED':False})
with ZipFile(BUNDLE_ZIP,'w',ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_ROOT.rglob('*')):
        if path.is_file(): archive.write(path,path.relative_to(OUTPUT_ROOT.parent))
    archive.write(SUBMISSION_ZIP,SUBMISSION_ZIP.name)
pipeline.close()
print({'B0_SAFE_SUBMISSION_VALIDATOR':RESULT['submission_validation'],'DOWNLOAD_SUBMISSION':str(SUBMISSION_ZIP),'DOWNLOAD_BUNDLE':str(BUNDLE_ZIP),'GT_OPENED':False})
